# Milestone 3 - Text Vectorization and Embeddings

This notebook implements the Milestone 3 text requirements from the capstone brief:

- build a review-text classifier with a bag-of-words / TF-IDF baseline;
- train a Keras neural model with learned text embeddings;
- compare baseline and embedding model with accuracy, macro-F1, and weighted-F1;
- build an embedding-based similar-products search demo using cosine similarity over product description embeddings.

The notebook preserves the frozen product-level train/validation/test split from earlier milestones so reviews of the same product cannot leak across splits.


## 0. Setup

The default neural-text run is intentionally moderate for a laptop. Increase `MAX_TEXT_TRAIN_ROWS`, `MAX_TEXT_VAL_ROWS`, or `MAX_TEXT_TEST_ROWS` before launching Jupyter for a fuller run.


In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.utils.class_weight import compute_class_weight

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODEL_DIR = PROJECT_ROOT / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_TRAIN_PATH = PROCESSED_DIR / "train.parquet"
REVIEW_VAL_PATH = PROCESSED_DIR / "val.parquet"
REVIEW_TEST_PATH = PROCESSED_DIR / "test.parquet"
PRODUCT_TRAIN_PATH = PROCESSED_DIR / "product_train.parquet"
PRODUCT_VAL_PATH = PROCESSED_DIR / "product_val.parquet"
PRODUCT_TEST_PATH = PROCESSED_DIR / "product_test.parquet"

TEXT_MODEL_PATH = MODEL_DIR / "text_embedding_classifier.keras"
TEXT_LABEL_ENCODER_PATH = MODEL_DIR / "text_label_encoder.joblib"
TFIDF_BASELINE_PATH = MODEL_DIR / "text_tfidf_logreg.joblib"
TEXT_METRICS_PATH = REPORTS_DIR / "milestone3_text_metrics.csv"
TEXT_CLASSIFICATION_REPORT_PATH = REPORTS_DIR / "milestone3_text_classification_report.csv"
TEXT_VAL_PREDICTIONS_PATH = PROCESSED_DIR / "text_embedding_predictions_val.csv"
TEXT_TEST_PREDICTIONS_PATH = PROCESSED_DIR / "text_embedding_predictions_test.csv"
SIMILAR_PRODUCTS_PATH = PROCESSED_DIR / "similar_products_demo.csv"
PRODUCT_EMBEDDINGS_PATH = PROCESSED_DIR / "product_text_embeddings.npy"
PRODUCT_SEARCH_INDEX_PATH = PROCESSED_DIR / "product_text_search_index.csv"

MAX_TEXT_TRAIN_ROWS = int(os.getenv("MAX_TEXT_TRAIN_ROWS", "20000"))
MAX_TEXT_VAL_ROWS = int(os.getenv("MAX_TEXT_VAL_ROWS", "5000"))
MAX_TEXT_TEST_ROWS = int(os.getenv("MAX_TEXT_TEST_ROWS", "5000"))
MAX_TFIDF_FEATURES = int(os.getenv("MAX_TFIDF_FEATURES", "30000"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "10000"))
SEQUENCE_LENGTH = int(os.getenv("SEQUENCE_LENGTH", "120"))
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "64"))
BATCH_SIZE = int(os.getenv("TEXT_BATCH_SIZE", "256"))
EPOCHS = int(os.getenv("TEXT_EPOCHS", "8"))
PATIENCE = int(os.getenv("TEXT_PATIENCE", "2"))

TARGET = "sentiment_label"
TEXT_COLUMN = "review_model_text"
BAND_ORDER = ["low", "medium", "high"]

MAX_SEARCH_PRODUCTS = int(os.getenv("MAX_SEARCH_PRODUCTS", "25000"))


## 1. Load Review and Product Splits

Milestone 3 text classification uses review text as the input and `sentiment_label` as the target. Because the review split was created by product, it is still safe for review-level text modeling.


In [ ]:
review_train = pd.read_parquet(REVIEW_TRAIN_PATH)
review_val = pd.read_parquet(REVIEW_VAL_PATH)
review_test = pd.read_parquet(REVIEW_TEST_PATH)
product_train = pd.read_parquet(PRODUCT_TRAIN_PATH)
product_val = pd.read_parquet(PRODUCT_VAL_PATH)
product_test = pd.read_parquet(PRODUCT_TEST_PATH)

required_review_columns = {"product_id", "review_text", "title", "clean_review_text", TARGET}
for split_name, df in {"train": review_train, "val": review_val, "test": review_test}.items():
    missing = required_review_columns - set(df.columns)
    if missing:
        raise ValueError(f"{split_name} review split is missing required columns: {sorted(missing)}")

print("Review splits:")
print("train", review_train.shape)
print("val", review_val.shape)
print("test", review_test.shape)

display(pd.concat({
    "train": review_train[TARGET].astype(str).value_counts(),
    "val": review_val[TARGET].astype(str).value_counts(),
    "test": review_test[TARGET].astype(str).value_counts(),
}, axis=1).fillna(0).astype(int))


## 2. Product-Level Leakage Check

The same product must not appear in more than one split. This protects both the review classifier and the product search evaluation examples from product leakage.


In [ ]:
train_ids = set(review_train["product_id"].dropna().astype(str))
val_ids = set(review_val["product_id"].dropna().astype(str))
test_ids = set(review_test["product_id"].dropna().astype(str))

leakage_summary = {
    "train_val_overlap": len(train_ids & val_ids),
    "train_test_overlap": len(train_ids & test_ids),
    "val_test_overlap": len(val_ids & test_ids),
}
leakage_check_passed = all(value == 0 for value in leakage_summary.values())

display(pd.DataFrame([leakage_summary]))
print(f"Leakage check passed: {leakage_check_passed}")
if not leakage_check_passed:
    raise ValueError("Product-level leakage detected across review splits.")


## 3. Prepare Review Text

The model text combines the review title and review body. Empty texts and missing targets are removed. Optional row caps keep the notebook quick while preserving the frozen split.


In [ ]:
def combine_review_text(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    title = out["title"].fillna("").astype(str)
    body = out["review_text"].fillna(out["clean_review_text"]).fillna("").astype(str)
    out[TEXT_COLUMN] = (title + " " + body).str.replace(r"\s+", " ", regex=True).str.strip()
    out[TARGET] = out[TARGET].astype(str)
    out = out[(out[TEXT_COLUMN] != "") & out[TARGET].notna()].copy()
    return out.reset_index(drop=True)


def stratified_cap(df: pd.DataFrame, max_rows: int) -> pd.DataFrame:
    if max_rows <= 0 or len(df) <= max_rows:
        return df.reset_index(drop=True)
    parts = []
    used_indices = []
    per_class = max(1, max_rows // df[TARGET].nunique())
    for _, group in df.groupby(TARGET, observed=True):
        n = min(len(group), per_class)
        sampled_group = group.sample(n=n, random_state=SEED)
        parts.append(sampled_group)
        used_indices.extend(sampled_group.index.tolist())
    sampled = pd.concat(parts)
    remaining = max_rows - len(sampled)
    if remaining > 0:
        pool = df.drop(index=used_indices, errors="ignore")
        if len(pool):
            sampled = pd.concat([sampled, pool.sample(n=min(remaining, len(pool)), random_state=SEED)])
    return sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

text_train = stratified_cap(combine_review_text(review_train), MAX_TEXT_TRAIN_ROWS)
text_val = stratified_cap(combine_review_text(review_val), MAX_TEXT_VAL_ROWS)
text_test = stratified_cap(combine_review_text(review_test), MAX_TEXT_TEST_ROWS)

for name, df in {"train": text_train, "val": text_val, "test": text_test}.items():
    print(name, df.shape)
    print(df[TARGET].value_counts().to_string())


## 4. Bag-of-Words / TF-IDF Baselines

The milestone asks for bag-of-words / TF-IDF first. The dummy classifier is a sanity baseline; logistic regression over TF-IDF is the main classical baseline.


In [ ]:
def evaluate_predictions(model_name: str, split_name: str, y_true, y_pred) -> dict:
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

metrics = []

X_train_text = text_train[TEXT_COLUMN].astype(str).to_numpy()
X_val_text = text_val[TEXT_COLUMN].astype(str).to_numpy()
X_test_text = text_test[TEXT_COLUMN].astype(str).to_numpy()
y_train_labels = text_train[TARGET].astype(str).to_numpy()
y_val_labels = text_val[TARGET].astype(str).to_numpy()
y_test_labels = text_test[TARGET].astype(str).to_numpy()

majority = DummyClassifier(strategy="most_frequent")
majority.fit(X_train_text.reshape(-1, 1), y_train_labels)
for split_name, X_eval, y_eval in [("val", X_val_text, y_val_labels), ("test", X_test_text, y_test_labels)]:
    pred = majority.predict(X_eval.reshape(-1, 1))
    metrics.append(evaluate_predictions("majority_baseline", split_name, y_eval, pred))

tfidf_logreg = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=MAX_TFIDF_FEATURES, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("logreg", OneVsRestClassifier(LogisticRegression(max_iter=500, class_weight="balanced", solver="liblinear", random_state=SEED))),
])
tfidf_logreg.fit(X_train_text, y_train_labels)

for split_name, X_eval, y_eval in [("val", X_val_text, y_val_labels), ("test", X_test_text, y_test_labels)]:
    pred = tfidf_logreg.predict(X_eval)
    metrics.append(evaluate_predictions("tfidf_logistic_regression", split_name, y_eval, pred))

pd.DataFrame(metrics).round(4)


## 5. Learned Embedding Text Classifier

This Keras model learns word embeddings from the review text, pools the token embeddings, and predicts sentiment. It is intentionally smaller than a transformer; Milestone 4 will handle transformer models.


In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_labels)
y_val = label_encoder.transform(y_val_labels)
y_test = label_encoder.transform(y_test_labels)
class_names = list(label_encoder.classes_)
num_classes = len(class_names)

tokenizer = Tokenizer(num_words=MAX_TOKENS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

def texts_to_padded_sequences(texts: np.ndarray | pd.Series) -> np.ndarray:
    sequences = tokenizer.texts_to_sequences(np.asarray(texts, dtype=object).astype(str))
    return pad_sequences(sequences, maxlen=SEQUENCE_LENGTH, padding="post", truncating="post")

X_train_seq = texts_to_padded_sequences(X_train_text)
X_val_seq = texts_to_padded_sequences(X_val_text)
X_test_seq = texts_to_padded_sequences(X_test_text)

class_weights_array = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=y_train)
class_weight = {idx: float(weight) for idx, weight in enumerate(class_weights_array)}
print("Classes:", class_names)
print("Sequence arrays:", X_train_seq.shape, X_val_seq.shape, X_test_seq.shape)
print("Class weights:", class_weight)


In [ ]:
def build_embedding_classifier(num_classes: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="token_ids")
    x = tf.keras.layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM, mask_zero=True, name="token_embedding")(inputs)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(96, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    x = tf.keras.layers.Dense(48, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.20)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="sentiment")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="review_text_embedding_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
        run_eagerly=True,
    )
    return model

embedding_model = build_embedding_classifier(num_classes)
embedding_model.summary()


In [ ]:
history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
best_val_loss = np.inf
best_weights = None
patience_counter = 0
rng = np.random.default_rng(SEED)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(reduction="none")
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
class_weight_tensor = tf.constant([class_weight[idx] for idx in range(num_classes)], dtype=tf.float32)

for epoch in range(EPOCHS):
    shuffled_idx = rng.permutation(len(X_train_seq))
    epoch_losses = []
    epoch_accuracies = []
    for start in range(0, len(shuffled_idx), BATCH_SIZE):
        batch_idx = shuffled_idx[start:start + BATCH_SIZE]
        xb = tf.convert_to_tensor(X_train_seq[batch_idx], dtype=tf.int32)
        yb = tf.convert_to_tensor(y_train[batch_idx], dtype=tf.int32)
        with tf.GradientTape() as tape:
            batch_proba = embedding_model(xb, training=True)
            sample_losses = loss_fn(yb, batch_proba)
            sample_weights = tf.gather(class_weight_tensor, yb)
            batch_loss = tf.reduce_mean(sample_losses * sample_weights)
        grads = tape.gradient(batch_loss, embedding_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, embedding_model.trainable_variables))
        batch_pred = tf.argmax(batch_proba, axis=1, output_type=tf.int32)
        batch_acc = tf.reduce_mean(tf.cast(tf.equal(batch_pred, yb), tf.float32))
        epoch_losses.append(float(batch_loss.numpy()))
        epoch_accuracies.append(float(batch_acc.numpy()))

    val_proba = embedding_model(tf.convert_to_tensor(X_val_seq, dtype=tf.int32), training=False)
    val_losses = tf.keras.losses.sparse_categorical_crossentropy(y_val, val_proba)
    val_loss = float(tf.reduce_mean(val_losses).numpy())
    val_pred = tf.argmax(val_proba, axis=1).numpy()
    val_accuracy = float(np.mean(val_pred == y_val))

    history["loss"].append(float(np.mean(epoch_losses)))
    history["accuracy"].append(float(np.mean(epoch_accuracies)))
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"loss: {history['loss'][-1]:.4f} - accuracy: {history['accuracy'][-1]:.4f} - "
        f"val_loss: {history['val_loss'][-1]:.4f} - val_accuracy: {history['val_accuracy'][-1]:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = embedding_model.get_weights()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping after epoch {epoch + 1}.")
            break

if best_weights is not None:
    embedding_model.set_weights(best_weights)

history = type("HistoryLike", (), {"history": history})()


## 6. Learning Curves


In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history_df["loss"], label="train")
axes[0].plot(history_df["val_loss"], label="validation")
axes[0].set_title("Embedding classifier loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["accuracy"], label="train")
axes[1].plot(history_df["val_accuracy"], label="validation")
axes[1].set_title("Embedding classifier accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()


## 7. Evaluation and Error Analysis


In [ ]:
def predict_label_names(model: tf.keras.Model, texts: np.ndarray) -> np.ndarray:
    sequences = texts_to_padded_sequences(texts)
    proba = model(tf.convert_to_tensor(sequences, dtype=tf.int32), training=False).numpy()
    return label_encoder.inverse_transform(np.argmax(proba, axis=1))

val_embedding_pred = predict_label_names(embedding_model, X_val_text)
test_embedding_pred = predict_label_names(embedding_model, X_test_text)

metrics.append(evaluate_predictions("keras_learned_embeddings", "val", y_val_labels, val_embedding_pred))
metrics.append(evaluate_predictions("keras_learned_embeddings", "test", y_test_labels, test_embedding_pred))

metrics_df = pd.DataFrame(metrics).sort_values(["split", "macro_f1"], ascending=[True, False]).reset_index(drop=True)
metrics_df.to_csv(TEXT_METRICS_PATH, index=False)
display(metrics_df.round(4))
print("Saved metrics:", TEXT_METRICS_PATH.relative_to(PROJECT_ROOT))


In [ ]:
report = classification_report(
    y_test_labels,
    test_embedding_pred,
    labels=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
report_df.to_csv(TEXT_CLASSIFICATION_REPORT_PATH)
display(report_df.round(4))
print("Saved classification report:", TEXT_CLASSIFICATION_REPORT_PATH.relative_to(PROJECT_ROOT))


In [ ]:
cm = confusion_matrix(y_test_labels, test_embedding_pred, labels=class_names)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title("Keras learned embeddings confusion matrix - test")
ax.set_xlabel("Predicted sentiment")
ax.set_ylabel("True sentiment")
plt.show()


In [ ]:
test_predictions = text_test[["product_id", "title", "review_text", TARGET, TEXT_COLUMN]].copy()
test_predictions["predicted_sentiment"] = test_embedding_pred
test_proba = embedding_model(tf.convert_to_tensor(X_test_seq, dtype=tf.int32), training=False).numpy()
for idx, label in enumerate(class_names):
    test_predictions[f"proba_{label}"] = test_proba[:, idx]
test_predictions["is_correct"] = test_predictions[TARGET].astype(str) == test_predictions["predicted_sentiment"].astype(str)

val_predictions = text_val[["product_id", "title", "review_text", TARGET, TEXT_COLUMN]].copy()
val_predictions["predicted_sentiment"] = val_embedding_pred
val_predictions.to_csv(TEXT_VAL_PREDICTIONS_PATH, index=False)
test_predictions.to_csv(TEXT_TEST_PREDICTIONS_PATH, index=False)

display(test_predictions.query("not is_correct").head(15))
print("Saved val predictions:", TEXT_VAL_PREDICTIONS_PATH.relative_to(PROJECT_ROOT))
print("Saved test predictions:", TEXT_TEST_PREDICTIONS_PATH.relative_to(PROJECT_ROOT))


## 8. Honest Baseline Check

The capstone brief asks the deep model to improve on the simple baseline. If the embedding model does not beat TF-IDF logistic regression, that is a valid result to report: TF-IDF is often very strong on small or moderately sized sentiment datasets because sentiment cues are direct lexical signals.


In [ ]:
comparison = metrics_df.pivot_table(index="model", columns="split", values="macro_f1", aggfunc="first")
logreg_val_macro_f1 = float(comparison.loc["tfidf_logistic_regression", "val"])
embedding_val_macro_f1 = float(comparison.loc["keras_learned_embeddings", "val"])
embedding_beats_logreg = embedding_val_macro_f1 > logreg_val_macro_f1

print(f"Embedding model beats TF-IDF Logistic Regression on validation macro-F1: {embedding_beats_logreg}")
print(f"TF-IDF Logistic Regression val macro-F1: {logreg_val_macro_f1:.4f}")
print(f"Keras learned embeddings val macro-F1: {embedding_val_macro_f1:.4f}")

if not embedding_beats_logreg:
    print("Honest interpretation: TF-IDF remains stronger on this split. Review sentiment is heavily driven by explicit words and phrases, so a linear model over bigrams can be highly competitive. The learned embedding model may need more data, pretrained embeddings, or a transformer to improve further.")


## 9. Build Product Description Text for Similar-Products Search

The search demo embeds product descriptions and returns nearest neighbors by cosine similarity. Product text combines title, store, category, and available metadata details from review-level rows.


In [ ]:
def safe_text(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

all_products = pd.concat([
    product_train.assign(split="train"),
    product_val.assign(split="val"),
    product_test.assign(split="test"),
], ignore_index=True)

review_product_details = pd.concat([review_train, review_val, review_test], ignore_index=True)
detail_cols = ["product_id", "details", "categories"]
available_detail_cols = [col for col in detail_cols if col in review_product_details.columns]
if available_detail_cols:
    product_details = review_product_details[available_detail_cols].drop_duplicates("product_id")
else:
    product_details = pd.DataFrame({"product_id": all_products["product_id"].unique()})

search_products = all_products.merge(product_details, on="product_id", how="left")
for col in ["product_title", "store_name", "main_category_clean", "details", "categories"]:
    if col not in search_products.columns:
        search_products[col] = ""

search_products["product_search_text"] = (
    safe_text(search_products["product_title"]) + " " +
    safe_text(search_products["store_name"]) + " " +
    safe_text(search_products["main_category_clean"]) + " " +
    safe_text(search_products["details"]) + " " +
    safe_text(search_products["categories"])
).str.strip()
search_products = search_products[search_products["product_search_text"] != ""].drop_duplicates("product_id").reset_index(drop=True)

if MAX_SEARCH_PRODUCTS > 0 and len(search_products) > MAX_SEARCH_PRODUCTS:
    full_search_products = search_products.copy()
    sampled_parts = []
    used_indices = []
    per_band = max(1, MAX_SEARCH_PRODUCTS // max(1, full_search_products["rating_band"].nunique()))
    for _, group in full_search_products.groupby("rating_band", observed=True):
        sampled_group = group.sample(n=min(len(group), per_band), random_state=SEED)
        sampled_parts.append(sampled_group)
        used_indices.extend(sampled_group.index.tolist())
    search_products = pd.concat(sampled_parts)
    remaining = MAX_SEARCH_PRODUCTS - len(search_products)
    if remaining > 0:
        pool = full_search_products.drop(index=used_indices, errors="ignore")
        if len(pool):
            search_products = pd.concat([search_products, pool.sample(n=min(remaining, len(pool)), random_state=SEED)])
    search_products = search_products.sample(frac=1.0, random_state=SEED).head(MAX_SEARCH_PRODUCTS).reset_index(drop=True)

print("Searchable products:", len(search_products))
display(search_products[["product_id", "product_title", "split", "rating_band", "product_search_text"]].head(5))


## 10. Product Text Embeddings

The product search embeddings reuse the learned token embedding layer from the review classifier. Each product description is vectorized, token embeddings are averaged with a mask, and the final vectors are L2-normalized for cosine search.


In [ ]:
embedding_layer = embedding_model.get_layer("token_embedding")


def embed_texts(texts: pd.Series | np.ndarray, batch_size: int = BATCH_SIZE) -> np.ndarray:
    sequence_array = texts_to_padded_sequences(np.asarray(texts, dtype=object).astype(str))
    vectors = []
    for start in range(0, len(sequence_array), batch_size):
        token_ids = tf.convert_to_tensor(sequence_array[start:start + batch_size])
        token_vectors = embedding_layer(token_ids)
        mask = tf.cast(tf.not_equal(token_ids, 0), tf.float32)[..., tf.newaxis]
        summed = tf.reduce_sum(token_vectors * mask, axis=1)
        counts = tf.maximum(tf.reduce_sum(mask, axis=1), 1.0)
        vectors.append((summed / counts).numpy())
    return normalize(np.vstack(vectors))

product_embeddings = embed_texts(search_products["product_search_text"])
np.save(PRODUCT_EMBEDDINGS_PATH, product_embeddings)
search_index = search_products[["product_id", "product_title", "store_name", "split", "rating_band", "product_search_text"]].copy()
search_index.to_csv(PRODUCT_SEARCH_INDEX_PATH, index=False)

print("Product embeddings:", product_embeddings.shape)
print("Saved embeddings:", PRODUCT_EMBEDDINGS_PATH.relative_to(PROJECT_ROOT))
print("Saved search index:", PRODUCT_SEARCH_INDEX_PATH.relative_to(PROJECT_ROOT))


## 11. Similar Products Demo

Use either a product ID or a free-text query. Product-ID search excludes the product itself from its results.


In [ ]:
def similar_products(query: str, top_k: int = 10) -> pd.DataFrame:
    query = str(query)
    product_match = search_index[search_index["product_id"].astype(str) == query]
    exclude_product_id = None
    if len(product_match):
        query_text = product_match.iloc[0]["product_search_text"]
        exclude_product_id = query
    else:
        query_text = query

    query_embedding = embed_texts(np.array([query_text]))
    scores = cosine_similarity(query_embedding, product_embeddings).ravel()
    results = search_index.copy()
    results["similarity"] = scores
    if exclude_product_id is not None:
        results = results[results["product_id"].astype(str) != exclude_product_id]
    return results.sort_values("similarity", ascending=False).head(top_k).reset_index(drop=True)

example_product_id = search_index.iloc[0]["product_id"]
product_query_results = similar_products(example_product_id, top_k=8)
free_text_query_results = similar_products("hydrating gentle cleanser for sensitive skin", top_k=8)

demo_results = pd.concat([
    product_query_results.assign(query_type="product_id", query=str(example_product_id)),
    free_text_query_results.assign(query_type="free_text", query="hydrating gentle cleanser for sensitive skin"),
], ignore_index=True)
demo_results.to_csv(SIMILAR_PRODUCTS_PATH, index=False)

display(demo_results[["query_type", "query", "product_id", "product_title", "split", "rating_band", "similarity"]])
print("Saved similar-products demo:", SIMILAR_PRODUCTS_PATH.relative_to(PROJECT_ROOT))


## 12. Save Models and Summary


In [ ]:
embedding_model.save(TEXT_MODEL_PATH)
joblib.dump(label_encoder, TEXT_LABEL_ENCODER_PATH)
joblib.dump(tfidf_logreg, TFIDF_BASELINE_PATH)

summary = {
    "text_task": "review sentiment classification",
    "search_task": "similar products by learned product text embeddings",
    "target": TARGET,
    "train_reviews": len(text_train),
    "val_reviews": len(text_val),
    "test_reviews": len(text_test),
    "searchable_products": len(search_index),
    "leakage_check": "PASS" if leakage_check_passed else "FAIL",
    "metrics_path": str(TEXT_METRICS_PATH.relative_to(PROJECT_ROOT)),
    "classification_report_path": str(TEXT_CLASSIFICATION_REPORT_PATH.relative_to(PROJECT_ROOT)),
    "model_path": str(TEXT_MODEL_PATH.relative_to(PROJECT_ROOT)),
    "similar_products_path": str(SIMILAR_PRODUCTS_PATH.relative_to(PROJECT_ROOT)),
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
print("Saved embedding model:", TEXT_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Saved TF-IDF baseline:", TFIDF_BASELINE_PATH.relative_to(PROJECT_ROOT))
print("Saved label encoder:", TEXT_LABEL_ENCODER_PATH.relative_to(PROJECT_ROOT))
